In [3]:
import cobra
import logging
import multiprocessing
from operator import attrgetter

import numpy as np
from cobra import Configuration, Reaction
from cobra.exceptions import Infeasible
from cobra.flux_analysis import flux_variability_analysis
from optlang.interface import INFEASIBLE, OPTIMAL
from optlang.symbolics import Zero

import memote.support.consistency_helpers as con_helpers
import memote.support.helpers as helpers

model= cobra.io.read_sbml_model('cow_add_information.xml')

In [4]:
def find_unconserved_metabolites(model):
    """
    Detect unconserved metabolites.

    Parameters
    ----------
    model : cobra.Model
        The metabolic model under investigation.

    Notes
    -----
    See [1]_ section 3.2 for a complete description of the algorithm.


    .. [1] Gevorgyan, A., M. G Poolman, and D. A Fell.
           "Detection of Stoichiometric Inconsistencies in Biomolecular
           Models."
           Bioinformatics 24, no. 19 (2008): 2245.

    """
    problem = model.problem
    stoich_trans = problem.Model()
    # We clone the existing configuration in order to apply non-default
    # settings, for example, for solver verbosity or timeout.
    stoich_trans.configuration = problem.Configuration.clone(
        config=model.solver.configuration, problem=stoich_trans
    )
    #internal_rxns= con_helpers.get_internals(model)
    internal_rxns = set([reaction for reaction in model.reactions if not reaction.boundary and not 'biomass' in reaction.name.lower()])
    metabolites = set(met for rxn in internal_rxns for met in rxn.metabolites)
    # The binary variables k[i] in the paper.
    k_vars = list()
    for met in metabolites:
        # The element m[i] of the mass vector.
        m_var = problem.Variable(met.id)
        k_var = problem.Variable("k_{}".format(met.id), type="binary")
        k_vars.append(k_var)
        stoich_trans.add([m_var, k_var])
        # This constraint is equivalent to 0 <= k[i] <= m[i].
        stoich_trans.add(
            problem.Constraint(k_var - m_var, ub=0, name="switch_{}".format(met.id))
        )
    stoich_trans.update()
    con_helpers.add_reaction_constraints(
        stoich_trans, internal_rxns, problem.Constraint
    )
    # The objective is to maximize the binary indicators k[i], subject to the
    # above inequality constraints.
    stoich_trans.objective = problem.Objective(Zero, sloppy=True, direction="max")
    stoich_trans.objective.set_linear_coefficients({var: 1.0 for var in k_vars})
    status = stoich_trans.optimize()
    if status == OPTIMAL:
        # TODO: See if that could be a Boolean test `bool(var.primal)`.
        return set(
            [
                model.metabolites.get_by_id(var.name[2:])
                for var in k_vars
                if var.primal < 0.8
            ]
        )
    else:
        raise RuntimeError(
            "Could not compute list of unconserved metabolites."
            " Solver status is '{}' (only optimal expected).".format(status))
        

In [5]:
find_unconserved_metabolites(model)

{<Metabolite MAM01266c at 0x7fb236608ee0>,
 <Metabolite MAM02039c at 0x7fb2363af4f0>,
 <Metabolite MAM02039e at 0x7fb2363af640>,
 <Metabolite MAM02039g at 0x7fb2363af520>,
 <Metabolite MAM02039i at 0x7fb235c5a370>,
 <Metabolite MAM02039l at 0x7fb2363af550>,
 <Metabolite MAM02039m at 0x7fb2363af580>,
 <Metabolite MAM02039n at 0x7fb2363af5b0>,
 <Metabolite MAM02039r at 0x7fb2363af610>,
 <Metabolite MAM02039x at 0x7fb2363af5e0>,
 <Metabolite MAM02933c at 0x7fb2360f4be0>,
 <Metabolite MAM02933e at 0x7fb235d5c0a0>,
 <Metabolite MAM02933r at 0x7fb236016a30>,
 <Metabolite MAM03357c at 0x7fb235f32670>}

In [6]:
# hydrogen imbalance 맞추기
available= ['charge', 'H']
for reaction in model.reactions:
    hydrogen_only_criteria= True
    if not reaction.boundary and int(reaction.id.replace('MAR',''))>13086 and not 'transport' in reaction.name.lower():
        for key in list(reaction.check_mass_balance().keys()):
            if key not in available:
                hydrogen_only_criteria= False
                break
        if not hydrogen_only_criteria:
            continue
        if 'H' in list(reaction.check_mass_balance().keys()):  
            metabolite_add= {'MAM02039c': -int(reaction.check_mass_balance()['H'])}
            reaction.add_metabolites(metabolite_add)

In [7]:
remove_reaction=[]
pass_metabolite= ['charge']
for reaction in model.reactions:
    if not reaction.boundary and int(reaction.id.replace('MAR',''))>13086 and not 'transport' in reaction.name.lower():
        for metabolite in reaction.check_mass_balance():
            if metabolite not in pass_metabolite:
                remove_reaction.append(reaction)
                break
for reaction in remove_reaction:
    model.reactions.get_by_id(reaction.id).remove_from_model()

In [5]:
criteria=True
while criteria==True:
    print('iteration start')
    criteria=False
    unconserved_len=dict()
    for reaction in remove_reaction:
        print(reaction.id+' start')
        reaction_name= reaction.id
        model.add_reactions([reaction])
        try:
            unconserved= find_unconserved_metabolites(model)
            print(len(unconserved))
            print('-------------------------')
            unconserved_len[reaction_name]= len(unconserved)
            model.reactions.get_by_id(reaction_name).remove_from_model()
        except:
            model.reactions.get_by_id(reaction_name).remove_from_model()
    for key,value in unconserved_len.items():
        if value==0:
            criteria=True
            reaction= [reaction for reaction in remove_reaction if reaction.id==key][0]
            model.add_reactions([reaction])
            remove_reaction.remove(reaction)
            break
    print('iteration ended')

iteration start
MAR13454 start
0
-------------------------
MAR13465 start
0
-------------------------
MAR13477 start
0
-------------------------
MAR13484 start
0
-------------------------
MAR13486 start
0
-------------------------
MAR13492 start
0
-------------------------
MAR13500 start
0
-------------------------
MAR13516 start
0
-------------------------
MAR13531 start
0
-------------------------
MAR13540 start
0
-------------------------
MAR13564 start
0
-------------------------
MAR13575 start
0
-------------------------
iteration ended
iteration start
MAR13465 start
0
-------------------------
MAR13477 start
0
-------------------------
MAR13484 start
0
-------------------------
MAR13486 start
0
-------------------------
MAR13492 start
0
-------------------------
MAR13500 start
0
-------------------------
MAR13516 start
0
-------------------------
MAR13531 start
0
-------------------------
MAR13540 start
0
-------------------------
MAR13564 start
0
-------------------------
MAR135

In [6]:
#각 metabolite의 mass 구하기 위함
problem = model.problem
stoich_trans = problem.Model()
# We clone the existing configuration in order to apply non-default
# settings, for example, for solver verbosity or timeout.
stoich_trans.configuration = problem.Configuration.clone(
    config=model.solver.configuration, problem=stoich_trans
)
#internal_rxns= con_helpers.get_internals(model)
internal_rxns = set([reaction for reaction in model.reactions if not reaction.boundary and not 'biomass' in reaction.name.lower()])
metabolites = set(met for rxn in internal_rxns for met in rxn.metabolites)
# The binary variables k[i] in the paper.
k_vars = list()
for met in metabolites:
    # The element m[i] of the mass vector.
    m_var = problem.Variable(met.id)
    k_var = problem.Variable("k_{}".format(met.id), type="binary")
    k_vars.append(k_var)
    stoich_trans.add([m_var, k_var])
    # This constraint is equivalent to 0 <= k[i] <= m[i].
    stoich_trans.add(
        problem.Constraint(k_var - m_var, ub=0, name="switch_{}".format(met.id))
    )
stoich_trans.update()
con_helpers.add_reaction_constraints(
    stoich_trans, internal_rxns, problem.Constraint
)
# The objective is to maximize the binary indicators k[i], subject to the
# above inequality constraints.
stoich_trans.objective = problem.Objective(Zero, sloppy=True, direction="max")
stoich_trans.objective.set_linear_coefficients({var: 1.0 for var in k_vars})
status = stoich_trans.optimize()

In [7]:
#MAR13500
print('reactant')
print(stoich_trans.primal_values['MAM02040c'])
print(stoich_trans.primal_values['MAM02684c'])
reactant_sum= int(stoich_trans.primal_values['MAM02040c'])+int((stoich_trans.primal_values['MAM02684c']))

print('product')
print(stoich_trans.primal_values['MAM01362c'])
print(stoich_trans.primal_values['MAM03813c'])
product_sum= int(stoich_trans.primal_values['MAM01362c'])+int((stoich_trans.primal_values['MAM03813c']))

print('product_sum- reactant_sum= {}'.format(product_sum-reactant_sum))

print('H2O')
print(stoich_trans.primal_values['MAM02040c'])


#MAR13516
print('reactant')
print(stoich_trans.primal_values['MAM02040c'])
print(stoich_trans.primal_values['MAM02684c'])
reactant_sum= int(stoich_trans.primal_values['MAM02040c'])+int((stoich_trans.primal_values['MAM02684c']))

print('product')
print(stoich_trans.primal_values['MAM02389c'])
print(stoich_trans.primal_values['MAM03813c'])
product_sum= int(stoich_trans.primal_values['MAM02389c'])+int((stoich_trans.primal_values['MAM03813c']))

print('product_sum- reactant_sum= {}'.format(product_sum-reactant_sum))

print('H2O')
print(stoich_trans.primal_values['MAM02040c'])

print('H')
print(stoich_trans.primal_values['MAM02039c'])

reactant
85.02100840321731
4224.050420160831
product
1857.462184870757
2617.6512604997247
product_sum- reactant_sum= 165
H+
85.02100840321731
reactant
85.02100840321731
4224.050420160831
product
1689.4201680643266
2617.6512604997247
product_sum- reactant_sum= -3
H2O
85.02100840321731
H
1.0000000000000782


In [8]:
#remove_reaction[0]= MAR13500
metabolite_add= {model.metabolites.get_by_id('MAM02040c'): -1}
remove_reaction[0].add_metabolites(metabolite_add)
model.add_reactions([remove_reaction[0]])

#remove_reaction[1]= MAR13516
metabolite_add= {model.metabolites.get_by_id('MAM02039c'): 2}
remove_reaction[1].add_metabolites(metabolite_add)
model.add_reactions([remove_reaction[1]])

In [9]:
#unconserved 확인
find_unconserved_metabolites(model)

set()

In [11]:
from pathlib import Path
from cobra.io import load_json_model, save_json_model, load_matlab_model, save_matlab_model, read_sbml_model, write_sbml_model
import logging

write_sbml_model(model, "cow_consistency.xml")